# FIFA World Cup Predictor

This notebook demonstrates the complete pipeline:
1. Load data (World Cup matches + international results)
2. Engineer features (ELO ratings, rolling stats, head-to-head)
3. Train XGBoost classifier
4. Predict match outcomes
5. Simulate 2026 World Cup tournament
6. Generate reports and visualizations

In [8]:
import warnings
warnings.filterwarnings('ignore')

# Reload modules to pick up code changes
import importlib
import sys
if 'predictor' in sys.modules:
    import predictor.config
    import predictor.data_loader
    import predictor.feature_engineer
    import predictor.model_trainer
    import predictor.predictor_api
    import predictor.simulator
    import predictor.reporter
    importlib.reload(predictor.config)
    importlib.reload(predictor.data_loader)
    importlib.reload(predictor.feature_engineer)
    importlib.reload(predictor.model_trainer)
    importlib.reload(predictor.predictor_api)
    importlib.reload(predictor.simulator)
    importlib.reload(predictor.reporter)

from predictor.config import DEFAULT_MODEL_PATH, DEFAULT_OUTPUT_DIR, RANDOM_SEED, FEATURE_COLS, OUTCOME_INT_MAP
from predictor.data_loader import DataLoader
from predictor.feature_engineer import FeatureEngineer
from predictor.model_trainer import ModelTrainer
from predictor.predictor_api import PredictorAPI
from predictor.simulator import TournamentSimulator
from predictor.reporter import ReportGenerator

DATA_DIR = '.'
CUTOFF_YEAR = 2014
N_RUNS = 1000
print('Imports OK')

Imports OK


## Step 1: Load Data

In [9]:
loader = DataLoader(DATA_DIR)
matches_df, players_df, cups_df, results_df = loader.load()
print(f'WC Matches:            {len(matches_df):>6} rows')
print(f'International results: {len(results_df):>6} rows  ({results_df["year"].min()}-{results_df["year"].max()})')
print(f'Players:               {len(players_df):>6} rows')
print(f'\nYear column info:')
print(f'  Type: {matches_df["Year"].dtype}')
print(f'  NaN count: {matches_df["Year"].isna().sum()}')
print(f'  Min: {matches_df["Year"].min()}, Max: {matches_df["Year"].max()}')
matches_df.head()

WorldCupMatches.csv: dropping 3720 row(s) with nulls in required columns.
WorldCupPlayers.csv: dropping 28715 row(s) with nulls in required columns.
results.csv: dropping 72 row(s) with nulls in required columns.


WC Matches:               836 rows
International results:  49215 rows  (1872-2026)
Players:                 9069 rows

Year column info:
  Type: int32
  NaN count: 0
  Min: 1930, Max: 2014


,Year,Datetime,Stage,Stadium,City,Home Team Name,Home Team Goals,Away Team Goals,Away Team Name,Win conditions,Attendance,Half-time Home Goals,Half-time Away Goals,Referee,Assistant 1,Assistant 2,RoundID,MatchID,Home Team Initials,Away Team Initials
0,1930,1930-07-13 15:00:00,Group 1,Pocitos,Montevideo,France,4.0,1.0,Mexico,,4444.0,3.0,0.0,LOMBARDI Domingo (URU),CRISTOPHE Henry (BEL),REGO Gilberto (BRA),201.0,1096.0,FRA,MEX
1,1930,1930-07-13 15:00:00,Group 4,Parque Central,Montevideo,USA,3.0,0.0,Belgium,,18346.0,2.0,0.0,MACIAS Jose (ARG),MATEUCCI Francisco (URU),WARNKEN Alberto (CHI),201.0,1090.0,USA,BEL
2,1930,1930-07-14 12:45:00,Group 2,Parque Central,Montevideo,Serbia,2.0,1.0,Brazil,,24059.0,2.0,0.0,TEJADA Anibal (URU),VALLARINO Ricardo (URU),BALWAY Thomas (FRA),201.0,1093.0,YUG,BRA
3,1930,1930-07-14 14:50:00,Group 3,Pocitos,Montevideo,Romania,3.0,1.0,Peru,,2549.0,1.0,0.0,WARNKEN Alberto (CHI),LANGENUS Jean (BEL),MATEUCCI Francisco (URU),201.0,1098.0,ROU,PER
4,1930,1930-07-15 16:00:00,Group 1,Parque Central,Montevideo,Argentina,1.0,0.0,France,,23409.0,0.0,0.0,REGO Gilberto (BRA),SAUCEDO Ulises (BOL),RADULESCU Constantin (ROU),201.0,1085.0,ARG,FRA


## Step 2: Feature Engineering

In [10]:
print('Building features (may take 1-2 minutes)...')
fe = FeatureEngineer(matches_df, players_df, results_df)
features_df = fe.build_features()
print(f'Feature matrix: {features_df.shape}')
print(f'Outcome distribution:')
print(features_df['outcome'].value_counts().to_string())
features_df.head()

Building features (may take 1-2 minutes)...
Feature matrix: (1672, 27)
Outcome distribution:
outcome
Home Win    650
Away Win    650
Draw        372


,focal_team,opponent,year,team_win_rate,team_draw_rate,team_loss_rate,team_avg_goals_scored,team_avg_goals_conceded,team_avg_goal_diff,opp_win_rate,...,h2h_avg_goal_diff,team_elo,opp_elo,elo_diff,stage_ordinal,tournament_weight,is_neutral,team_avg_goal_events_per_player,opp_avg_goal_events_per_player,outcome
0,France,Mexico,1930,0.310345,0.103448,0.586207,1.873563,3.149425,-1.275862,0.500000,...,NaN,1359.493100,1489.293758,-129.800657,1,1.0,1,0.0,0.0,Home Win
1,Mexico,France,1930,0.500000,0.125000,0.375000,2.125000,2.500000,-0.375000,0.310345,...,NaN,1489.293758,1359.493100,129.800657,1,1.0,1,0.0,0.0,Away Win
2,USA,Belgium,1930,0.466667,0.133333,0.400000,2.333333,3.200000,-0.866667,0.386139,...,-7.0,1489.974707,1473.541460,16.433247,1,1.0,1,0.0,0.0,Home Win
3,Belgium,USA,1930,0.386139,0.148515,0.465347,2.128713,2.306931,-0.178218,0.466667,...,7.0,1473.541460,1489.974707,-16.433247,1,1.0,1,0.0,0.0,Away Win
4,Serbia,Brazil,1930,0.312500,0.062500,0.625000,1.718750,3.156250,-1.437500,0.421053,...,NaN,1459.223807,1552.103749,-92.879942,1,1.0,1,0.0,0.0,Home Win


## Step 3: Train Model

In [11]:
trainer = ModelTrainer(features_df, cutoff_year=CUTOFF_YEAR, model_path=DEFAULT_MODEL_PATH)
result = trainer.train()
trainer.save()
print('Test set metrics:')
for k, v in result.metrics.items():
    print(f'  {k}: {v:.4f}')
print(f'\nTop 10 features by importance:')
print(result.feature_importance.nlargest(10).to_string())

Test set metrics:
  accuracy: 0.6068
  precision: 0.5688
  recall: 0.6068
  f1: 0.5491

Top 10 features by importance:
elo_diff                  0.184801
team_elo                  0.070024
opp_elo                   0.068171
year                      0.048723
h2h_avg_goal_diff         0.048245
opp_avg_goals_scored      0.047435
stage_ordinal             0.043573
team_avg_goals_scored     0.042689
h2h_team_win_rate         0.038885
opp_avg_goals_conceded    0.038506


## Step 4: Test Predictions

In [12]:
api = PredictorAPI(result.model, fe)

test_pairs = [
    ('Brazil', 'Germany'),
    ('France', 'Argentina'),
    ('Spain', 'England'),
]
print('Sample match predictions:')
for home, away in test_pairs:
    try:
        pred = api.predict(home, away)
        print(f'{home:15} vs {away:15}  |  Home: {pred.home_win_prob:.3f}  Draw: {pred.draw_prob:.3f}  Away: {pred.away_win_prob:.3f}  =>  {pred.predicted_label}')
    except ValueError as e:
        print(f'  Skipping: {e}')

Sample match predictions:
Brazil          vs Germany          |  Home: 0.409  Draw: 0.258  Away: 0.333  =>  Home Win
France          vs Argentina        |  Home: 0.341  Draw: 0.296  Away: 0.363  =>  Away Win
Spain           vs England          |  Home: 0.387  Draw: 0.267  Away: 0.346  =>  Home Win


## Step 5: Setup 2026 World Cup Groups

In [13]:
# Official 2026 FIFA World Cup groups (draw held December 5, 2025)
# 48-team tournament across 12 groups
WC_2026_GROUPS = {
    'A': ['Mexico', 'South Africa', 'South Korea', 'Czech Republic'],
    'B': ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland'],
    'C': ['Brazil', 'Morocco', 'Haiti', 'Scotland'],
    'D': ['USA', 'Paraguay', 'Australia', 'Turkey'],
    'E': ['Germany', 'Curacao', 'Ivory Coast', 'Ecuador'],
    'F': ['Netherlands', 'Japan', 'Sweden', 'Tunisia'],
    'G': ['Belgium', 'Egypt', 'Iran', 'New Zealand'],
    'H': ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay'],
    'I': ['France', 'Senegal', 'Norway', 'Iraq'],
    'J': ['Argentina', 'Algeria', 'Austria', 'Jordan'],
    'K': ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia'],
    'L': ['England', 'Croatia', 'Ghana', 'Panama'],
}

known = set(api._team_lookup.keys())
valid_groups = {}
for g, teams in WC_2026_GROUPS.items():
    valid = [t for t in teams if t.lower() in known]
    if len(valid) >= 2:
        valid_groups[g] = valid

valid_teams = [t for teams in valid_groups.values() for t in teams]
print(f'Valid groups: {len(valid_groups)}, teams: {len(valid_teams)}')
for g, t in valid_groups.items():
    print(f'  Group {g}: {t}')

Valid groups: 12, teams: 47
  Group A: ['Mexico', 'South Africa', 'South Korea', 'Czech Republic']
  Group B: ['Canada', 'Bosnia and Herzegovina', 'Qatar', 'Switzerland']
  Group C: ['Brazil', 'Morocco', 'Haiti', 'Scotland']
  Group D: ['USA', 'Paraguay', 'Australia', 'Turkey']
  Group E: ['Germany', 'Ivory Coast', 'Ecuador']
  Group F: ['Netherlands', 'Japan', 'Sweden', 'Tunisia']
  Group G: ['Belgium', 'Egypt', 'Iran', 'New Zealand']
  Group H: ['Spain', 'Cape Verde', 'Saudi Arabia', 'Uruguay']
  Group I: ['France', 'Senegal', 'Norway', 'Iraq']
  Group J: ['Argentina', 'Algeria', 'Austria', 'Jordan']
  Group K: ['Portugal', 'DR Congo', 'Uzbekistan', 'Colombia']
  Group L: ['England', 'Croatia', 'Ghana', 'Panama']


## Step 6: Simulate Tournament & Generate Reports

In [14]:
if len(valid_groups) >= 12:
    simulator = TournamentSimulator(api, n_runs=N_RUNS, seed=RANDOM_SEED)
    sim_result = simulator.simulate(valid_teams, valid_groups)

    reporter = ReportGenerator(DEFAULT_OUTPUT_DIR)
    reporter.bar_chart(sim_result)
    reporter.feature_importance(result.feature_importance)
    reporter.summary_csv(sim_result)

    # Confusion matrix on test set
    test_df = features_df[features_df['year'] >= CUTOFF_YEAR]
    y_true = test_df['outcome'].tolist()
    y_pred_int = result.model.predict(test_df[list(FEATURE_COLS)])
    y_pred = [OUTCOME_INT_MAP[i] for i in y_pred_int]
    reporter.confusion_matrix(y_true, y_pred)

    print('\nTop 10 predicted 2026 WC winners:')
    for i, team in enumerate(sim_result.ranked_teams[:10], 1):
        prob = sim_result.win_probabilities[team]
        sf_prob = sim_result.semifinal_probabilities[team]
        print(f'  {i:2d}. {team:<20} win={prob:.3f}  semi={sf_prob:.3f}')
else:
    print(f'Only {len(valid_groups)} valid groups — need 12. Check team name mapping.')
    print('Groups missing teams from training data:')
    for g, teams in WC_2026_GROUPS.items():
        missing = [t for t in teams if t.lower() not in known]
        if missing:
            print(f'  Group {g}: {missing}')


Top 10 predicted 2026 WC winners:
   1. Argentina            win=0.062  semi=0.182
   2. Spain                win=0.052  semi=0.182
   3. France               win=0.047  semi=0.177
   4. England              win=0.046  semi=0.165
   5. Germany              win=0.043  semi=0.162
   6. Netherlands          win=0.043  semi=0.149
   7. Portugal             win=0.043  semi=0.154
   8. Brazil               win=0.039  semi=0.145
   9. Ecuador              win=0.037  semi=0.108
  10. Turkey               win=0.036  semi=0.163
